In [1]:
from matplotlib import pyplot as plt
import numpy as np

from tqdm import tqdm

import time

from PieceDetection import PieceDetection

from Dataset.DataSetLoaders import ChessDataset

/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
ds = ChessDataset.ChessDataset(
    config={
        "img_size": (640,640)
    }
)

In [3]:
pc_cnn = PieceDetection.PieceDetector("cnn")
pc_yolo = PieceDetection.PieceDetector("yolo")

In [4]:
accs = []
avg_time = 0
for img, label in tqdm(ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_cnn.preprocess()
        preds = pc_cnn.predict()
        end_time = time.perf_counter()
        avg_time += end_time - start_time

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass

avg_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Processing Time: {avg_time*1e3:.3f}ms")

100%|██████████| 389/389 [00:25<00:00, 15.32it/s]

Acc: 0.96 | Errors: 2.7712 | Avg Processing Time: 53.697ms


In [5]:
accs = []
avg_time = 0
for img, label in tqdm(ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_yolo.preprocess()
        preds = pc_yolo.predict()
        end_time = time.perf_counter()
        avg_time += end_time - start_time
        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass


avg_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Processing Time: {avg_time*1e3:.3f}ms")

100%|██████████| 389/389 [00:30<00:00, 12.65it/s]

Acc: 0.99 | Errors: 0.6581 | Avg Processing Time: 53.926ms


In [5]:
train_ds, valid_ds, test_ds = ChessDataset.ChessDataset.train_valid_test_split(ds, sizes=(.8,.1,.1), random_state=42)

In [6]:
accs = []
for img, label in tqdm(test_ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        pc_cnn.preprocess()
        preds = pc_cnn.predict()

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass

accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f}")

100%|██████████| 40/40 [00:02<00:00, 14.82it/s]

Acc: 0.92 | Errors: 5.1500


In [11]:
accs = []
for img, label in tqdm(test_ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        pc_yolo.preprocess()
        preds = pc_yolo.predict()

        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass

accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f}")

100%|██████████| 40/40 [00:03<00:00, 12.71it/s]

Acc: 0.99 | Errors: 0.4000
